# Extended Data

In [1]:
import ee
import geemap
from utils import *
initialize()

config = ProjectConfig()
roi = config.roi
data_folder = config.data_folder
last_year = config.last_year

mapbiomas, lulc = desired_lulc()

In [ ]:

def fc_to_image(fc, property, scale = 1000):
    # Convert the feature collection to an image
    return fc.reduceToImage(
            properties = [property],
            reducer = ee.Reducer.first()
        ).reproject(
            crs = 'EPSG:4326',  # or match your source CRS
            scale = scale
        ).rename(property)

def export_image(image, description):
    task = ee.batch.Export.image.toDrive(
        image = image,
        description = description,
        fileNamePrefix = description,
        region = roi,
        scale = image.projection().nominalScale(),
        maxPixels = 1e13,
        crs = 'EPSG:4326',
        fileFormat = 'GeoTIFF'
    )
    task.start()

## Mature Forest

Get the mature forest by distance to edge.

In [ ]:
biomes = ee.Image(f"{data_folder}/categorical").select("biome")
biomes_mask = biomes.eq(1).rename("biome_mask")

lulc = (ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1")
            .select([f"classification_{year}" for year in config.range_1985_2020])
            .byte()
            .rename([str(year) for year in config.range_1985_2020]))

mature_mask = lulc.eq(3).reduce(ee.Reducer.allNonZero()).selfMask().updateMask(biomes_mask)

distance_forest_edge = ee.Image(f"{data_folder}/distance_forest_edge")

biomass_raw = (ee.Image(f"projects/sat-io/open-datasets/ESA/ESA_CCI_AGB/CCI_BIOMASS_100m_AGB_{last_year}_v51").select("AGB").rename(f"ESA_CCI_{last_year}"))


## Export EU TMF data for comparison with MapBiomas

In [ ]:
# Load the image collections
transition = ee.ImageCollection('projects/JRC/TMF/v1_2023/TransitionMap_Subtypes').mosaic().clip(roi)
annual_changes = ee.ImageCollection('projects/JRC/TMF/v1_2023/AnnualChanges').mosaic().clip(roi)

# Define regrowth and degraded conditions
regrowth = transition.gte(31).And(transition.lte(33))

# Initialize AgeRegrowth and AgeDegraded
tmf = ee.Image.constant(0)

# Calculate AgeRegrowth
for i in range(1990, last_year):
    year = 'Dec' + str(i)
    annual_changes_year = annual_changes.select(year)
    condition = annual_changes_year.eq(4).And(regrowth) # were regrowing then AND are regrowing now
    tmf = tmf.add(condition.eq(1))

tmf = tmf.selfMask().rename(f"tmf_{last_year}")


ESA_CCI = ee.ImageCollection("projects/sat-io/open-datasets/ESA/ESA_CCI_AGB").filterDate('2020-01-01','2021-01-01').select("AGB").mean().rename("biomass")

ESA_CCI_resampled = ESA_CCI.reduceResolution(
        reducer= ee.Reducer.mean(),
    ).reproject(
        crs=tmf.projection(),
        scale=tmf.projection().nominalScale()
    )

tmf_mask = tmf.gt(0).selfMask().rename("tmf_mask")

tmf_ESA = ESA_CCI.addBands(tmf).addBands(tmf_mask)


tmf_ESA_fc = tmf_ESA.stratifiedSample(numPoints = 100,
                                      classBand = "tmf",
                                      dropNulls = True)

task = ee.batch.Export.table.toDrive(collection = tmf_ESA_fc, fileFormat="CSV")
# task.start()


## Export asymptotes for Figure 2

In [ ]:
nearest_mature = ee.Image(f"{data_folder}/nearest_mature").selfMask()

quarters_ecoreg_biomass = ee.Image(f"{data_folder}/quarters_ecoreg_biomass").select("quarter_biomass").selfMask()

biomes = ee.Image(f"{data_folder}/categorical").select("biome")
biomes_mask = biomes.eq(1).rename("biome_mask").selfMask()

# nearest_mature = ee.Image(f"{data_folder}/nearest_mature").selfMask()


# export_image(nearest_mature, "nearest_mature")
# export_image(quarters_ecoreg_biomass, "quarters_ecoreg_biomass")
# export_image(biomes_mask, "biome_mask")

## Export predictions for Figure 4

In [ ]:
# image = ee.Image(f"{data_folder}/mapbiomas_2020").select("age")

image = ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_secondary_vegetation_age_v1").select("secondary_vegetation_age_2020")


biomes = ee.Image(f"{config.data_folder}/categorical").select("biome")


biomes = biomes.eq(1).selfMask()

pixels_to_sample = biomes.reduceResolution(
    ee.Reducer.first(), maxPixels=65536
    ).reproject(
    crs = image.projection().getInfo()['crs'],
    crsTransform = image.projection().getInfo()['transform']
    ).updateMask(image)

image_scale = round(pixels_to_sample.projection().nominalScale().getInfo())
closest_multiple = round(1000 / image_scale) * image_scale

# First, sample locations based only on the age band
grid = geemap.create_grid(pixels_to_sample.geometry(), closest_multiple, pixels_to_sample.projection())


# Predictions for 2050
pred_lag_2050_fc = ee.FeatureCollection(f"{data_folder}/results/pred_2050_secondary")

# pred_lag_2050 = fc_to_image(pred_lag_2050_fc, "pred").reproject(crs = pixels_to_sample.projection().getInfo()['crs'] , crsTransform = pixels_to_sample.projection().atScale(990).getInfo()['transform'])

pred_lag_2050 = pred_lag_2050_fc.reduceToImage(properties=['pred'], reducer=ee.Reducer.first()).reproject(crs = pixels_to_sample.projection().getInfo()['crs'] , crsTransform = pixels_to_sample.projection().atScale(990).getInfo()['transform'])

secondary_area = ee.Image("projects/amazon-forest-regrowth/assets/secondary_area_1km")

# # Aggregate the high-resolution pixels into the 10 km grid
# pred_lag_2050_10k = pred_lag_2050.reduceResolution(
#     reducer = ee.Reducer.median(),
#     maxPixels = 65535,
#     bestEffort = True
# ).reproject(
#     crs = 'EPSG:4326',
#     crsTransform=age.projection().atScale(990).getInfo()['transform'],
# ).rename("pred_lag_2050_10k")

vis_params = {'min': 0, 'max': 75, 'palette': ["#003f5c", "#2f4b7c", "#665191", "#a05195",
    "#d45087", "#f95d6a", "#ff7c43", "#ffa600", "#ffc300", "#ffda6a"]}

# map = geemap.Map()
# # map.addLayer(pred_lag_2050_fc, {}, "pred_lag_2050_fc")
# # map.addLayer(grid, {}, "grid")
# # map.addLayer(pred_lag_2050, vis_params, "biomass_2050")
# map.addLayer(pred_lag_2050, vis_params, "pred_lag_2050")
# # map.addLayer(secondary_area, {}, "secondary_area")
# map.addLayer(pred_lag_2050_interpolated, vis_params, "pred_lag_2050_interpolated")
# # map.add_colorbar(vis_params, label = "biomass (Mg/ha)", layer_name = "biomass")
# map

# export_image(pred_lag_2050_10k, "pred_2050_secondary")

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [ ]:
image = ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_secondary_vegetation_age_v1").select("secondary_vegetation_age_2020")

age = ee.Image(f"{data_folder}/mapbiomas_2020")


vis_params = {'min': 0, 'max': 35, 'palette': ["#003f5c", "#2f4b7c", "#665191", "#a05195",
    "#d45087", "#f95d6a", "#ff7c43", "#ffa600", "#ffc300", "#ffda6a"]}


map = geemap.Map()
map.addLayer(image, vis_params, "age_original")
map.addLayer(age, vis_params, "age_new")
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [ ]:
lulc = (ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1")
            .select([f"classification_{year}" for year in config.range_1985_2020])
            .byte()
            .rename([str(year) for year in config.range_1985_2020]))

map.addLayer(lulc, {}, "LULC")

In [ ]:
pred_lag_2050_fc = ee.FeatureCollection("projects/amazon-forest-regrowth/assets/pred_2050_secondary")
pred_lag_2050 = fc_to_image(pred_lag_2050_fc, "pred", scale=100)

# Aggregate the high-resolution pixels into the 10 km grid
pred_lag_2050 = pred_lag_2050.reduceResolution(
    reducer = ee.Reducer.median(),
    maxPixels = 65535,
    bestEffort = True
).reproject(
    crs = 'EPSG:4326',
    scale = 10000
).rename("pred_lag_2050_10k")

export_image(pred_lag_2050, "pred_lag_2050_secondary")

In [ ]:
pred_lag_2050_fc = ee.FeatureCollection("projects/amazon-forest-regrowth/assets/pred_2050_pastureland_all")




pred_lag_2050 = fc_to_image(pred_lag_2050_fc, "pred", scale=100)

# Aggregate the high-resolution pixels into the 10 km grid
pred_lag_2050 = pred_lag_2050.reduceResolution(
    reducer = ee.Reducer.median(),
    maxPixels = 65535,
    bestEffort = True
).reproject(
    crs = 'EPSG:4326',
    scale = 10000
).rename("pred_lag_2050_10k")

export_image(pred_lag_2050, "pred_lag_2050_pastureland_all_10k_raw")

In [ ]:
mature_biomass = ee.Image(f"{data_folder}/mature_biomass")

# Aggregate the high-resolution pixels into the 10 km grid
mature_biomass = mature_biomass.reduceResolution(
    reducer = ee.Reducer.median(),
    maxPixels = 65535,
    bestEffort = True
).reproject(
    crs = 'EPSG:4326',
    scale = 10000
).rename("mature_biomass")

export_image(mature_biomass, "mature_biomass")

In [ ]:

# Predictions for 2050
pred_lag_2050_fc = ee.FeatureCollection(f"{data_folder}/total_pred_lag_2050_secondary")
# multiply the value of pred by 1000
pred_lag_2050 = fc_to_image(pred_lag_2050_fc, "pred", scale=100)

# multiply value by 1000
# pred_lag_2050 = pred_lag_2050.multiply(1000)

# Aggregate the high-resolution pixels into the 10 km grid
pred_lag_2050_10k = pred_lag_2050.reduceResolution(
    reducer = ee.Reducer.median(),
    maxPixels = 65535,
    bestEffort = True
).reproject(
    crs = 'EPSG:4326',
    scale = 10000
).rename("pred_lag_2050_10k")


export_image(pred_lag_2050_10k, "total_pred_lag_2050_secondary")